In [1]:
import sys
sys.path.append("..")

import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import train_test_split

from src.data import load_data
from src.features import CATEGORICAL_COLS, get_X_y
from src.model import RANDOM_STATE

bundle = joblib.load("../models/model.joblib")
model = bundle["model"]

df_model, _ = load_data("../data/data.csv")
X, y = get_X_y(df_model)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [2]:
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

def transform(X):
    X_t = preprocessor.transform(X)
    return X_t.toarray() if hasattr(X_t, "toarray") else X_t

X_train_t = transform(X_train)
X_test_t = transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print(X_train_t.shape)
print(feature_names[:5])

(2904, 220)
['cat__marital_status_1' 'cat__marital_status_2' 'cat__marital_status_3'
 'cat__marital_status_4' 'cat__marital_status_5']


In [3]:
explainer = shap.LinearExplainer(classifier, X_train_t)
shap_values = explainer(X_test_t)

Background dataset has 2904 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=2904 when initializing the masker.


In [4]:
i = 0
log_odds = shap_values.base_values[i] + shap_values.values[i].sum()
prob_from_shap = 1 / (1 + np.exp(-log_odds))
prob_from_model = model.predict_proba(X_test.iloc[[i]])[0, 1]

print(prob_from_shap, prob_from_model)

0.012328732338008089 0.012328732338008089


In [5]:
def original_feature(name):
    prefix, rest = name.split("__", 1)
    if prefix == "cat":
        for col in CATEGORICAL_COLS:
            if rest.startswith(col + "_"):
                return col
    return rest

original_names = [original_feature(n) for n in feature_names]

def explain_student(i):
    contributions = pd.Series(shap_values.values[i], index=original_names)
    return contributions.groupby(level=0).sum().sort_values()

explain_student(0)

curricular_units_1st_sem_approved              -0.778288
course                                         -0.777326
scholarship_holder                             -0.582335
curricular_units_1st_sem_enrolled              -0.382865
curricular_units_1st_sem_evaluations           -0.255255
debtor                                         -0.238493
curricular_units_1st_sem_credited              -0.234339
curricular_units_1st_sem_grade                 -0.216009
application_mode                               -0.208869
mother_s_occupation                            -0.160391
age_at_enrollment                              -0.140028
gender                                         -0.120137
admission_grade                                -0.119060
unemployment_rate                              -0.107161
mother_s_qualification                         -0.066740
father_s_qualification                         -0.016090
inflation_rate                                 -0.011845
nationality                    

In [6]:
dropout_idx = np.where(y_test.values == 1)[0]
explain_student(dropout_idx[0])

curricular_units_1st_sem_enrolled              -3.761087
course                                         -1.499742
application_mode                               -1.209476
curricular_units_1st_sem_evaluations           -0.835378
unemployment_rate                              -0.461414
debtor                                         -0.238493
curricular_units_1st_sem_credited              -0.234339
mother_s_occupation                            -0.160391
age_at_enrollment                              -0.110794
admission_grade                                -0.069083
inflation_rate                                 -0.068703
father_s_qualification                         -0.016090
nationality                                    -0.003867
educational_special_needs                       0.000000
international                                   0.004428
previous_qualification                          0.013294
marital_status                                  0.015662
curricular_units_1st_sem_withou

In [7]:
X_test.iloc[dropout_idx[0]][[
    "curricular_units_1st_sem_enrolled",
    "curricular_units_1st_sem_approved",
    "curricular_units_1st_sem_grade",
]]

curricular_units_1st_sem_enrolled    0.0
curricular_units_1st_sem_approved    0.0
curricular_units_1st_sem_grade       0.0
Name: 1558, dtype: float64